# Construction du modele (role de l'Analyst)
Ce notebook lit les transactions dans **HDFS**, construit un modele **Spark ML**, le journalise dans **MLflow** et l'enregistre dans le **registre**.
Ensuite, le Senior Analyst attribue un stage (`staging` puis `preprod`) avec `scripts/promote_model.py` ou dans l'interface MLflow.

> Les donnees existent seulement apres avoir lance le generateur et laisse tourner le streaming quelques minutes.

In [ ]:
import os, mlflow, mlflow.spark
from pyspark.sql import SparkSession, functions as F

os.environ.setdefault("HADOOP_USER_NAME", "root")
mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "http://mlflow:5000"))
mlflow.set_experiment("fraud-detection-notebook")

spark = SparkSession.builder.master("local[2]").appName("model-building").getOrCreate()
df = spark.read.parquet("hdfs://namenode:9000/lake/transactions/")
print(df.count(), "transactions")
df.groupBy("is_fraud").count().show()

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

data = df.withColumn("label", F.col("is_fraud").cast("double"))
train, test = data.randomSplit([0.8, 0.2], seed=42)

# Ponderation des classes (peu de fraudes)
ratio = train.filter("label = 0").count() / max(train.filter("label = 1").count(), 1)
train = train.withColumn("w", F.when(F.col("label") == 1, ratio).otherwise(1.0))

stages = [
    StringIndexer(inputCol="merchant", outputCol="merchant_idx", handleInvalid="keep"),
    OneHotEncoder(inputCols=["merchant_idx"], outputCols=["merchant_oh"]),
    VectorAssembler(inputCols=["merchant_oh", "log_amount", "amount", "hour", "is_night", "is_foreign", "device_new"],
                    outputCol="features"),
    RandomForestClassifier(labelCol="label", weightCol="w", numTrees=50, maxDepth=6, seed=42),
]
pipeline = Pipeline(stages=stages)

In [ ]:
with mlflow.start_run(run_name="notebook_random_forest") as run:
    model = pipeline.fit(train)
    pred = model.transform(test)
    auc_pr = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderPR").evaluate(pred)
    auc_roc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC").evaluate(pred)
    mlflow.log_params({"numTrees": 50, "maxDepth": 6})
    mlflow.log_metrics({"auc_pr": auc_pr, "auc_roc": auc_roc})
    # Log du modele dans le registre MLflow
    mlflow.spark.log_model(model, "model", registered_model_name="fraud-detector")
print(f"AUC-PR = {auc_pr:.3f} | AUC-ROC = {auc_roc:.3f}")

## Etape suivante : attribuer un stage (Senior Analyst)
Dans un terminal :
```
docker exec airflow python /opt/project/scripts/promote_model.py --alias staging
docker exec airflow python /opt/project/scripts/promote_model.py --alias preprod
```
Ou dans MLflow (http://localhost:5000) : *Models* -> `fraud-detector` -> version -> *Aliases*.

In [ ]:
spark.stop()